<a href="https://colab.research.google.com/github/dogor97/acta_ENSSTC/blob/main/Codigo_acta_ENSSTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install docxtpl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#import docxtpl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dir = "/content/drive/MyDrive/Normal/notas/notas 2do periodo septimo 2026.xlsx"
dir2 = "/content/drive/MyDrive/Normal/notas/notas_original.xlsx"
dir3 = "/content/drive/MyDrive/Normal/notas/notas_test1.xlsx"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Clase lectora documento

In [ ]:
class XLSFormatter:
    def __init__(self, dir, numsheet):
        self.dir = dir
        self.numsheet = numsheet
        self.grades = ["PREESCOLAR", "PRIMERO", "SEGUNDO", "TERCERO", "CUARTO",
                       "QUINTO", "SEXTO", "SÉPTIMO", "OCTAVO", "NOVENO",
                       "DÉCIMO", "ONCE"]
        self.gradesnum = ["UNO", "DOS", "TRES", "CUATRO", "CINCO"]
        self.period = ["1/4", "2/4", "3/4", "4/4"]
        self.namesubj = ["NOMBRE", "Ciencias naturales", "Ciencias sociales y cátedra de la paz",
                         "Comportamiento", "Educación artística",
                         "Educación física", "Ética", "Informática", "Inglés",
                         "Investigación", "Español", "Matemáticas", "Pedagogía", "Religión"]
        self.cat = ["SUPERIOR", "ALTO", "BASICO", "BAJO"]
        self.gradeid = None
        self.gradenumid = None
        self.periodid = None
        self.rsumx = None
        self.bestx = None
        self.dfmpx = None
        self.dfepx = None
        self.notasup = 50
        self.notaalto = 45
        self.notabasico = 40
        self.notaperder = 30

    def formatxlsx(self):

        #Leer el excel
        rxlsx = pd.read_excel(self.dir, sheet_name=self.numsheet-1, header=None).drop([0,1]).reset_index(drop=True)
        self.gradeid = [i for i, num in enumerate(self.grades) if num in rxlsx.iloc[0,0].split()][0]
        self.gradenumid = [i for i, num in enumerate(self.gradesnum) if num in rxlsx.iloc[0, 0].split()][0]
        self.periodid = [i for i in self.period if i in rxlsx.iloc[2, 0].split()][0]

        rxlsx = rxlsx.drop([0, 1, 2, 3]).reset_index(drop=True)

        #Filtrar las columnas
        column_to_keep_index = rxlsx.iloc[1][rxlsx.iloc[1].str.contains(self.periodid, na=False)].index.tolist()
        columns_to_drop = [col for col in range(1, len(rxlsx.columns)) if col not in column_to_keep_index]
        rxlsx.drop(columns=columns_to_drop, inplace=True)
        rxlsx = rxlsx.drop(rxlsx.index[1])
        rxlsx.iloc[0] = self.namesubj
        rxlsx = rxlsx.T.set_index(rxlsx.T.iloc[:, 0]).drop(columns=rxlsx.columns[0]).T
        rxlsx.set_index(rxlsx.iloc[:, 0], inplace=True)
        rxlsx.drop(columns=rxlsx.columns[0], inplace=True)

        #Mover Comportamiento a la ultima columna
        column_to_move = rxlsx.pop("Comportamiento")
        rxlsx["Comportamiento"] = column_to_move

        return rxlsx

    def notas_sum(self, dfx):
        sumx = pd.DataFrame(0, index=self.cat, columns=dfx.columns)

        # Sumatorio por materias (Superior, Alto, Basico y Bajo)
        for index, row in dfx.iterrows():
            for i in dfx.columns:
                if row[i] >= self.notaalto and row[i] <= self.notasup: sumx[i][0] += 1
                if row[i] >= self.notabasico and row[i] < self.notaalto: sumx[i][1] += 1
                if row[i] >= self.notaperder and row[i] < self.notabasico: sumx[i][2] += 1
                if row[i] < self.notaperder: sumx[i][3] += 1

        sumx = sumx.T
        self.rsumx = sumx.sort_index().reindex(index=sumx.index.tolist()[:-1] + ['Comportamiento'])

        # Mejores 3 estudiantes
        dfx_mean = dfx.drop('Comportamiento', axis=1)
        dfx_mean["Promedio"] = dfx_mean.mean(axis=1)
        dfx_mean["Promedio"] = dfx_mean["Promedio"].astype(float)
        self.bestx = dfx_mean.nlargest(3, ["Promedio"])

        # Estudiantes que perdieron por materia
        perdidos_por_materia = dfx.apply(lambda col: col[dfx[col.name] < self.notaperder].index.tolist()).to_dict()
        perdidos_count = {subject: len(students) for subject, students in perdidos_por_materia.items()}
        self.dfmpx = pd.DataFrame({"Perdidos por materia": perdidos_count, "ALUMNOS": perdidos_por_materia})

        # Materias perdidas por estudiante
        dfex = dfx.drop('Comportamiento', axis=1)[dfx.lt(self.notaperder).any(axis=1)]
        self.dfepx = pd.DataFrame({
            'CANTIDAD': dfex.lt(self.notaperder).sum(axis=1),
            'MATERIAS': dfex.apply(lambda row: row.index[row.lt(self.notaperder)].tolist(), axis=1)})

        return self.rsumx, self.bestx, self.dfmpx, self.dfepx

# Uso de la clase

In [ ]:
m1m1 = XLSFormatter(dir3, 1)
m1m2 = XLSFormatter(dir3, 2)
m1m3 = XLSFormatter(dir3, 3)

In [ ]:
m1 = m1m1.formatxlsx()
m2 = m1m2.formatxlsx()
m3 = m1m3.formatxlsx()

In [ ]:
rsumx1, bestx1, dfmpx1, dfepx1 = m1m1.notas_sum(m1)
rsumx2, bestx2, dfmpx2, dfepx2 = m1m2.notas_sum(m2)
rsumx3, bestx3, dfmpx3, dfepx3 = m1m3.notas_sum(m3)

In [ ]:
m1m1 = XLSFormatter(dir, 1)
m1 = m1m1.formatxlsx()
rsumx1, bestx1, dfmpx1, dfepx1 = m1m1.notas_sum(m1)

/tmp/ipykernel_826/2576460211.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[i] >= self.notaperder and row[i] < self.notabasico: sumx[i][2] += 1
/tmp/ipykernel_826/2576460211.py:61: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and e

# Creador de listas para acta ENSSTC

### Septimo 1


In [ ]:
rsumx1

,SUPERIOR,ALTO,BASICO,BAJO
0,,,,
Ciencias naturales,2,8,19,5
Ciencias sociales y cátedra de la paz,0,0,33,1
Educación artística,5,9,19,1
Educación física,15,6,13,0
Ética,9,10,12,3
Informática,0,0,34,0
Inglés,0,8,26,0
Investigación,0,0,33,1
Español,1,14,19,0


In [ ]:
bestx1

,Ciencias naturales,Ciencias sociales y cátedra de la paz,Educación artística,Educación física,Ética,Informática,Inglés,Investigación,Español,Matemáticas,Pedagogía,Religión,Promedio
NOMBRE,,,,,,,,,,,,,
"BAUTISTA RONDON , YOSMAN SANTIAGO",43,35,45,49,50,36,43,38,44,35,40,45,41.916667
"CABRERA ROMERO, LYCETH DAYANA",46,37,42,48,41,34,42,38,44,39,44,47,41.833333
"ORTIZ CÁCERES, YOJAN ALEXIS",45,37,41,48,46,36,41,38,45,37,35,41,40.833333


In [ ]:
dfmpx1

,Perdidos por materia,ALUMNOS
Ciencias naturales,5,"[CALDERON VASQUEZ , FREDDY ALONSO, CARVAJAL ..."
Ciencias sociales y cátedra de la paz,1,"[MORA CASTRO, GLENDER DAVID]"
Educación artística,1,"[CONDE ABRIL, ANDREY STIVEN]"
Educación física,0,[]
Ética,3,"[MORA CASTRO, GLENDER DAVID, RANGEL CUADROS, E..."
Informática,0,[]
Inglés,0,[]
Investigación,1,"[CONDE ABRIL, ANDREY STIVEN]"
Español,0,[]
Matemáticas,5,"[ALARCÓN BASTO, DAYRON FABIÁN, BOHORQ..."


In [ ]:
dfepx1

,CANTIDAD,MATERIAS
NOMBRE,,
"ALARCÓN BASTO, DAYRON FABIÁN",1,[Matemáticas]
"BOHORQUEZ ARIAS, JHOJAN STEEVEN",1,[Matemáticas]
"CALDERON CALDERON, JHON ALEX",1,[Pedagogía]
"CALDERON VASQUEZ , FREDDY ALONSO",2,"[Ciencias naturales, Pedagogía]"
"CARVAJAL ANTOLÍNEZ, OSCAR MANUEL",2,"[Ciencias naturales, Matemáticas]"
"CASTELLANOS NARANJO, EMERSON YESID",2,"[Ciencias naturales, Matemáticas]"
"CONDE ABRIL, ANDREY STIVEN",3,"[Educación artística, Investigación, Pedagogía]"
"MORA CASTRO, GLENDER DAVID",4,"[Ciencias naturales, Ciencias sociales y cáted..."
"RANGEL CUADROS, EVER JESUS",1,[Ética]


### Septimo 2

In [ ]:
rsumx2

In [ ]:
bestx2

In [ ]:
dfmpx2

In [ ]:
dfepx2

### Septimo 3

In [ ]:
rsumx3

In [ ]:
bestx3

In [ ]:
dfmpx3

In [ ]:
dfepx3

# Solo un grado

In [ ]:
#Sumatorias
m1m1 = XLSFormatter(dir, 1)
m1 = m1m1.formatxlsx()
rsumx1, bestx1, dfmpx1, dfepx1 = m1m1.notas_sum(m1)

/tmp/ipykernel_6691/2576460211.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[i] >= self.notaperder and row[i] < self.notabasico: sumx[i][2] += 1
/tmp/ipykernel_6691/2576460211.py:61: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and

In [ ]:
rsumx1

,SUPERIOR,ALTO,BASICO,BAJO
0,,,,
Ciencias naturales,2,8,19,5
Ciencias sociales y cátedra de la paz,0,0,33,1
Educación artística,5,9,19,1
Educación física,15,6,13,0
Ética,9,10,12,3
Informática,0,0,34,0
Inglés,0,8,26,0
Investigación,0,0,33,1
Español,1,14,19,0


In [ ]:
bestx1

,Ciencias naturales,Ciencias sociales y cátedra de la paz,Educación artística,Educación física,Ética,Informática,Inglés,Investigación,Español,Matemáticas,Pedagogía,Religión,Promedio
NOMBRE,,,,,,,,,,,,,
"BAUTISTA RONDON , YOSMAN SANTIAGO",43,35,45,49,50,36,43,38,44,35,40,45,41.916667
"CABRERA ROMERO, LYCETH DAYANA",46,37,42,48,41,34,42,38,44,39,44,47,41.833333
"ORTIZ CÁCERES, YOJAN ALEXIS",45,37,41,48,46,36,41,38,45,37,35,41,40.833333


In [ ]:
dfmpx1

,Perdidos por materia,ALUMNOS
Ciencias naturales,5,"[CALDERON VASQUEZ , FREDDY ALONSO, CARVAJAL ..."
Ciencias sociales y cátedra de la paz,1,"[MORA CASTRO, GLENDER DAVID]"
Educación artística,1,"[CONDE ABRIL, ANDREY STIVEN]"
Educación física,0,[]
Ética,3,"[MORA CASTRO, GLENDER DAVID, RANGEL CUADROS, E..."
Informática,0,[]
Inglés,0,[]
Investigación,1,"[CONDE ABRIL, ANDREY STIVEN]"
Español,0,[]
Matemáticas,5,"[ALARCÓN BASTO, DAYRON FABIÁN, BOHORQ..."


In [ ]:
dfepx1

,CANTIDAD,MATERIAS
NOMBRE,,
"ALARCÓN BASTO, DAYRON FABIÁN",1,[Matemáticas]
"BOHORQUEZ ARIAS, JHOJAN STEEVEN",1,[Matemáticas]
"CALDERON CALDERON, JHON ALEX",1,[Pedagogía]
"CALDERON VASQUEZ , FREDDY ALONSO",2,"[Ciencias naturales, Pedagogía]"
"CARVAJAL ANTOLÍNEZ, OSCAR MANUEL",2,"[Ciencias naturales, Matemáticas]"
"CASTELLANOS NARANJO, EMERSON YESID",2,"[Ciencias naturales, Matemáticas]"
"CONDE ABRIL, ANDREY STIVEN",3,"[Educación artística, Investigación, Pedagogía]"
"MORA CASTRO, GLENDER DAVID",4,"[Ciencias naturales, Ciencias sociales y cáted..."
"RANGEL CUADROS, EVER JESUS",1,[Ética]
